# Acciones bélicas en Colombia · Cuaderno técnico

**Pregunta.** ¿Cómo cambian los patrones de las acciones bélicas registradas según el periodo, el territorio, la modalidad y la presencia de víctimas?

Trabajo de **Análisis y Visualización de Datos**, Universidad Nacional de Colombia. Equipo académico: Manuela Zapata Jiménez y David Cardona Duque. Fuente: Observatorio de Memoria y Conflicto del CNMH, corte del 9 de febrero de 2026.

[Resumen de resultados](../analisis.ipynb) · [Fuente de datos del CNMH](https://micrositios.centrodememoriahistorica.gov.co/observatorio/portal-de-datos/el-conflicto-en-cifras/acciones-belicas/)

**Cómo leerlo.** Las celdas ejecutables comprueban las tablas del repositorio y la regla de fechas. Las secciones siguientes conservan código de la entrega académica para inspección. Su ejecución integral requeriría el corte original, su GeoJSON y el entorno con pandas, NumPy, SciPy, statsmodels, scikit-learn, Matplotlib y Plotly.

## 1. Calidad de fechas y cobertura

La tabla permite separar los eventos con fecha completa, los eventos que solo conservan un año utilizable y los que carecen de un año válido. Mantener esa distinción evita perder registros en el análisis anual.

In [1]:
from csv import DictReader
from pathlib import Path

raiz = Path.cwd()
if raiz.name == 'tecnico':
    raiz = raiz.parent

def leer_tabla(nombre):
    with (raiz / 'datos' / nombre).open(encoding='utf-8', newline='') as archivo:
        return list(DictReader(archivo))

calidad = {fila['indicador']: int(fila['registros']) for fila in leer_tabla('calidad_temporal.csv')}
total = calidad['Fecha completa válida'] + calidad['Sin fecha completa']
assert total == 38_333
assert 0 < calidad['Año cero o nulo'] <= calidad['Sin fecha completa']
print(f"Registros con año utilizable: {total - calidad['Año cero o nulo']:,}")
print(f"Registros con fecha completa válida: {calidad['Fecha completa válida']:,}")

Registros con año utilizable: 38,113
Registros con fecha completa válida: 37,212


**Regla de calendario en ejecución.** Esta adaptación corta reproduce la distinción con un ejemplo inventado. Un registro puede conservar un año para los conteos aunque su día sea imposible en ese mes.

In [2]:
from datetime import date

def clasificar_fecha(anio, mes, dia):
    if not isinstance(anio, int) or not 1 <= anio <= 9999:
        return 'Sin año utilizable'
    if mes is None or dia is None:
        return 'Año utilizable, fecha incompleta'
    try:
        date(anio, mes, dia)
    except ValueError:
        return 'Año utilizable, fecha inválida'
    return 'Fecha completa válida'

fechas_inventadas = [(2000, 2, 29), (2001, 2, 29), (2001, None, None), (0, 1, 1)]
estados = [clasificar_fecha(*valores) for valores in fechas_inventadas]
assert estados == ['Fecha completa válida', 'Año utilizable, fecha inválida',
                   'Año utilizable, fecha incompleta', 'Sin año utilizable']
for campos, estado in zip(fechas_inventadas, estados):
    print(campos, estado)

(2000, 2, 29) Fecha completa válida
(2001, 2, 29) Año utilizable, fecha inválida
(2001, None, None) Año utilizable, fecha incompleta
(0, 1, 1) Sin año utilizable


## 2. Preparación de los registros

La entrega transforma respuestas como «Sin información» en faltantes y conserva el año por separado. Se revisan duplicados, tipos y categorías antes de construir la fecha. Los gráficos usan grupos de registros y una unidad de análisis consistente.

**Conversión de tipos y construcción inicial de variables**

```python
# ── 1. Renombrar columnas de muertes para claridad semántica ─────────────────
RENAME_MUERTES = {
    'Militares':                'Muertes Militares',
    'Policías':                 'Muertes Policías',
    'Otras Fuerzas Armadas Estatales': 'Muertes Otras Fuerzas Armadas Estatales',
    'Agentes del Estado Sin Información': 'Muertes Agentes del Estado Sin Información',
    'Total Agentes del Estado': 'Muertes Total Agentes del Estado',
    'Guerrilleros':             'Muertes Guerrilleros',
    'Paramilitares':            'Muertes Paramilitares',
    'Grupos Posdesmovilización':'Muertes Grupos Posdesmovilización',
    'Combatientes Sin Información': 'Muertes Combatientes Sin Información',
    'Otros Grupos Armados Organizados': 'Muertes Otros Grupos Armados Organizados',
    'Total Combatientes de Grupos Armados Organizados': 'Muertes Total Combatientes GAO',
    'Total Combatientes':       'Muertes Total Combatientes',
    'Personas Sin Información': 'Muertes Personas Sin Información',
    'Total Civiles':            'Muertes Total Civiles',
}
df.rename(columns=RENAME_MUERTES, inplace=True)
print(f"Columnas renombradas ({len(RENAME_MUERTES)}):")
for old, new in RENAME_MUERTES.items():
    print(f"  {old:<55} → {new}")

# ── 2. Tipos string ─────────────────────────────────────────────────────────
str_cols = ['ID Caso', 'ID Caso Relacionado', 'Código DANE de Municipio',
            'Municipio', 'Departamento', 'Región', 'latitud-longitud']
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).replace('nan', pd.NA)

# ── 3. Tipos entero (Int64 nullable) ────────────────────────────────────────
int_cols = [
    'Año', 'Mes', 'Día',
    'Lesionados Civiles', 'Capturados', 'Lesionados Combatientes',
    'Total de Víctimas del Caso',
] + list(RENAME_MUERTES.values())

for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

# ── 4. Tipos categóricos ────────────────────────────────────────────────────
cat_cols = [
    'Modalidad', 'Iniciativa', 'Tipo de Unidad Atacada',
    'Grupo Armado 1', 'Descripción Grupo Armado 1',
    'Grupo Armado 2', 'Descripción Grupo Armado 2',
    'Grupo Armado 3', 'Descripción Grupo Armado 3',
    'Grupo al que Pertenecen los Capturados', 'Ventaja Militar',
]
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

# ── 5. Extracción de coordenadas ────────────────────────────────────────────
def parse_point(val):
    if pd.isna(val):
        return pd.Series([np.nan, np.nan])
    try:
        coords = str(val).replace('POINT (', '').replace(')', '').strip().split()
        lon, lat = float(coords[0]), float(coords[1])
        return pd.Series([lat, lon])
    except Exception:
        return pd.Series([np.nan, np.nan])

if 'latitud-longitud' in df.columns:
    df[['Latitud', 'Longitud']] = df['latitud-longitud'].apply(parse_point)

# ── 6. Variables derivadas ──────────────────────────────────────────────────
df['Año_int'] = df['Año'].fillna(0).astype(int)

bins_decada = list(range(1950, 2031, 10))
labels_decada = [f"{d}s" for d in bins_decada[:-1]]
df['Década'] = pd.cut(df['Año_int'], bins=bins_decada, labels=labels_decada, right=False)

df['Tiene_Víctimas'] = (df['Total de Víctimas del Caso'] > 0).astype(int)

# ── Resumen ─────────────────────────────────────────────────────────────────
print(f"\nTransformaciones completadas:")
print(f"  - Columnas renombradas (muertes):   {len(RENAME_MUERTES)}")
print(f"  - Columnas → string:                {len(str_cols)}")
print(f"  - Columnas → Int64:                 {len(int_cols)}")
print(f"  - Columnas → category:              {len(cat_cols)}")
print(f"  - Coordenadas extraídas:            Latitud, Longitud")
print(f"  - Variables derivadas:              Año_int, Década, Tiene_Víctimas")
print(f"  - Dimensiones finales:              {df.shape[0]:,} filas × {df.shape[1]} columnas")

print("\nVerificación de tipos de datos:")
print(df.dtypes.to_string())
```

**Diagnóstico de faltantes**

```python
print("=" * 80)
print("1. ANÁLISIS DE VALORES FALTANTES (COMPLETITUD)")
print("=" * 80)

null_summary = pd.DataFrame({
    'Columna': df.columns,
    'Nulos': df.isna().sum().values,
    '% Nulos': (df.isna().mean() * 100).round(2).values,
    'No Nulos': df.notna().sum().values,
}).sort_values('% Nulos', ascending=False)

null_summary = null_summary[null_summary['Nulos'] > 0]
null_summary.index = range(1, len(null_summary) + 1)
print(f"\nColumnas con valores faltantes: {len(null_summary)} de {len(df.columns)}")
print(f"Registros completamente llenos: {df.dropna().shape[0]:,} de {len(df):,} ({df.dropna().shape[0]/len(df)*100:.1f}%)")
null_summary
```

**Limpieza de convenciones de ausencia**

```python
print("=" * 80)
print("1.1. TRATAMIENTO DE VALORES FALTANTES")
print("=" * 80)

df_prep = df.copy()

# ── 1. Unificar "SIN INFORMACION" → NaN en columnas categóricas y de texto ──
sin_info_values = ['SIN INFORMACION', 'SIN INFORMACIÓN', 'NO APLICA', 'N/A', 'NA']
cols_sin_info = ['Departamento', 'Municipio', 'Región', 'Modalidad',
                 'Grupo Armado 1', 'Descripción Grupo Armado 1',
                 'Grupo Armado 2', 'Descripción Grupo Armado 2',
                 'Grupo Armado 3', 'Descripción Grupo Armado 3',
                 'Iniciativa', 'Ventaja Militar', 'Tipo de Unidad Atacada',
                 'Grupo al que Pertenecen los Capturados']

n_replaced_total = 0
print("\n--- Reemplazos 'SIN INFORMACIÓN' → NaN ---")
for col in cols_sin_info:
    if col in df_prep.columns:
        mask = df_prep[col].isin(sin_info_values)
        n_replaced = mask.sum()
        if n_replaced > 0:
            df_prep.loc[mask, col] = pd.NA
            print(f"  {col:<45} → {n_replaced:>6,} valores reemplazados")
            n_replaced_total += n_replaced

print(f"\n  Total de valores 'SIN INFORMACIÓN' unificados a NaN: {n_replaced_total:,}")

# ── 2. Variables temporales: valor 0 → NaN ───────────────────────────────────
print("\n--- Tratamiento de variables temporales con valor 0 ---")
for col in ['Año', 'Mes', 'Día']:
    if col in df_prep.columns:
        mask_zero = df_prep[col] == 0
        n_zero = mask_zero.sum()
        if n_zero > 0:
            df_prep.loc[mask_zero, col] = pd.NA
            print(f"  {col}: {n_zero:,} registros con valor 0 → NaN")

if 'Año_int' in df_prep.columns:
    df_prep['Año_int'] = df_prep['Año'].fillna(0).astype(int)

# ── 3. Resumen post-tratamiento ──────────────────────────────────────────────
print("\n--- Completitud tras tratamiento de faltantes ---")
null_after = df_prep.isna().sum()
total_cells = df_prep.shape[0] * df_prep.shape[1]
total_nulls_after = null_after.sum()
print(f"  Completitud general: {(1 - total_nulls_after / total_cells) * 100:.1f}%")
print(f"  (Nota: la completitud disminuye porque los 'SIN INFORMACIÓN' ahora se contabilizan como nulos reales)")
```

**Validación del día, mes y año**

```python
print("=" * 80)
print("3.2. CREACIÓN DE VARIABLE FECHA")
print("=" * 80)

has_year = df_prep['Año_int'] > 0
has_month = df_prep['Mes'].notna() & (df_prep['Mes'] > 0) & (df_prep['Mes'] <= 12)
has_day = df_prep['Día'].notna() & (df_prep['Día'] > 0) & (df_prep['Día'] <= 31)

mask_complete = has_year & has_month & has_day

df_prep['Fecha'] = pd.NaT

date_strings = (
    df_prep.loc[mask_complete, 'Año_int'].astype(str) + '-' +
    df_prep.loc[mask_complete, 'Mes'].astype(int).astype(str).str.zfill(2) + '-' +
    df_prep.loc[mask_complete, 'Día'].astype(int).astype(str).str.zfill(2)
)

df_prep.loc[mask_complete, 'Fecha'] = pd.to_datetime(date_strings, format='%Y-%m-%d', errors='coerce')

n_with_date = df_prep['Fecha'].notna().sum()
n_without_date = df_prep['Fecha'].isna().sum()

print(f"\n  Registros con fecha completa válida: {n_with_date:,} ({n_with_date/len(df_prep)*100:.1f}%)")
print(f"  Registros sin fecha completa:        {n_without_date:,} ({n_without_date/len(df_prep)*100:.1f}%)")

print("\n  Rango temporal del dataset:")
valid_dates = df_prep['Fecha'].dropna()
if len(valid_dates) > 0:
    print(f"    Fecha más antigua: {valid_dates.min().strftime('%Y-%m-%d')}")
    print(f"    Fecha más reciente: {valid_dates.max().strftime('%Y-%m-%d')}")

print(f"\n  Motivos de ausencia de fecha:")
print(f"    Año = 0 o nulo:    {(~has_year).sum():,} registros")
print(f"    Mes = 0 o nulo:    {(~has_month).sum():,} registros")
print(f"    Día = 0 o nulo:    {(~has_day).sum():,} registros")
```

## 3. Modalidades, actores y presencia de víctimas

La V de Cramér resume la fuerza de asociaciones categóricas. El análisis original obtuvo aproximadamente **0,44** para iniciativa y **0,36** para modalidad frente a la presencia de víctimas. La prueba de Kruskal–Wallis dio **H = 5.026,41** al comparar la distribución del número de víctimas entre modalidades. La salida numérica redondeó el valor p a cero por la precisión de cómputo. La asociación no identifica causas.

**Tablas de contingencia y V de Cramér**

```python
from scipy.stats import chi2_contingency

def cramers_v(var1, var2):
    confusion_matrix = pd.crosstab(var1, var2)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    min_dim = min(confusion_matrix.shape) - 1
    if min_dim == 0 or n == 0:
        return 0.0
    return np.sqrt(chi2 / (n * min_dim))

cat_vars_analysis = ['Modalidad', 'Grupo Armado 1', 'Grupo Armado 2',
                     'Iniciativa', 'Ventaja Militar', 'Departamento']

df_cat_clean = df_prep[cat_vars_analysis].dropna()

print("=" * 80)
print("1.1.2. ASOCIACIÓN ENTRE VARIABLES CATEGÓRICAS (CHI² y CRAMÉR'S V)")
print("=" * 80)

n_vars = len(cat_vars_analysis)
cramer_matrix = pd.DataFrame(np.zeros((n_vars, n_vars)),
                              index=cat_vars_analysis, columns=cat_vars_analysis)
pvalue_matrix = pd.DataFrame(np.ones((n_vars, n_vars)),
                              index=cat_vars_analysis, columns=cat_vars_analysis)

for i in range(n_vars):
    for j in range(i, n_vars):
        if i == j:
            cramer_matrix.iloc[i, j] = 1.0
            pvalue_matrix.iloc[i, j] = 0.0
        else:
            v1 = df_cat_clean[cat_vars_analysis[i]]
            v2 = df_cat_clean[cat_vars_analysis[j]]
            ct = pd.crosstab(v1, v2)
            chi2, p, dof, expected = chi2_contingency(ct)
            cv = cramers_v(v1, v2)
            cramer_matrix.iloc[i, j] = cv
            cramer_matrix.iloc[j, i] = cv
            pvalue_matrix.iloc[i, j] = p
            pvalue_matrix.iloc[j, i] = p

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(cramer_matrix, dtype=bool), k=1)
sns.heatmap(cramer_matrix.astype(float), mask=mask, annot=True, fmt='.3f', cmap='YlOrRd',
            vmin=0, vmax=1, square=True, linewidths=0.5,
            cbar_kws={'label': "V de Cramér"}, ax=ax)
ax.set_title("Asociación entre variables categóricas (V de Cramér)", fontsize=14, pad=20)
plt.tight_layout()
save_fig(fig, 'fase4_cramers_v_matrix')
plt.show()

alpha = 0.05
n_tests = n_vars * (n_vars - 1) // 2
alpha_bonferroni = alpha / n_tests
print(f"\nCorrección de Bonferroni: α ajustado = {alpha}/{n_tests} = {alpha_bonferroni:.6f}")

print(f"\nAsociaciones significativas (p < {alpha_bonferroni:.6f}):")
for i in range(n_vars):
    for j in range(i+1, n_vars):
        p = pvalue_matrix.iloc[i, j]
        cv = cramer_matrix.iloc[i, j]
        sig = "✓" if p < alpha_bonferroni else "✗"
        if p < alpha_bonferroni:
            strength = "fuerte" if cv > 0.3 else "moderada" if cv > 0.1 else "débil"
            print(f"  {sig} {cat_vars_analysis[i]} ↔ {cat_vars_analysis[j]}: "
                  f"V = {cv:.3f} ({strength}), p = {p:.2e}")
```

**Asociación de predictores con el indicador de víctimas**

```python
print("=" * 80)
print("1.1.3. ASOCIACIÓN DE VARIABLES CATEGÓRICAS CON Tiene_Víctimas")
print("=" * 80)

target = 'Tiene_Víctimas'
cat_predictors = ['Modalidad', 'Grupo Armado 1', 'Grupo Armado 2',
                  'Iniciativa', 'Ventaja Militar', 'Departamento', 'Década']

results_cat = []
for col in cat_predictors:
    if col not in df_prep.columns:
        continue
    sub = df_prep[[col, target]].dropna()
    ct = pd.crosstab(sub[col], sub[target])
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        continue
    chi2, p, dof, expected = chi2_contingency(ct)
    cv = cramers_v(sub[col], sub[target])
    results_cat.append({'Variable': col, 'Chi²': chi2, 'p-valor': p,
                        'DoF': dof, "V de Cramér": cv})

results_cat_df = pd.DataFrame(results_cat).sort_values("V de Cramér", ascending=False)
results_cat_df['Significativa'] = results_cat_df['p-valor'] < alpha_bonferroni
display(results_cat_df.reset_index(drop=True))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71' if s else '#e74c3c' for s in results_cat_df['Significativa']]
bars = ax.barh(results_cat_df['Variable'], results_cat_df["V de Cramér"], color=colors)
ax.set_xlabel("V de Cramér")
ax.set_title(f"Fuerza de asociación con '{target}' (verde = significativa, rojo = no)", fontsize=13)
ax.axvline(x=0.1, color='gray', linestyle='--', alpha=0.5, label='Umbral débil (0.1)')
ax.axvline(x=0.3, color='gray', linestyle=':', alpha=0.5, label='Umbral moderado (0.3)')
ax.legend()
plt.tight_layout()
save_fig(fig, 'fase4_cramers_v_vs_target')
plt.show()

print("\nInterpretación:")
for _, row in results_cat_df.iterrows():
    strength = "fuerte" if row["V de Cramér"] > 0.3 else "moderada" if row["V de Cramér"] > 0.1 else "débil"
    sig = "significativa" if row['Significativa'] else "no significativa"
    print(f"  {row['Variable']}: asociación {strength} ({sig}), V = {row['V de Cramér']:.3f}")
```

**Kruskal–Wallis y comparaciones por modalidad**

```python
from scipy.stats import kruskal, mannwhitneyu

print("=" * 80)
print("1.3.2. PRUEBA DE KRUSKAL-WALLIS: VÍCTIMAS POR MODALIDAD")
print("=" * 80)

groups_mod = []
labels_mod = []
for mod, group in df_prep.groupby('Modalidad')['Total de Víctimas del Caso']:
    data = group.dropna().values
    if len(data) >= 5:
        groups_mod.append(data)
        labels_mod.append(mod)

stat_kw, p_kw = kruskal(*groups_mod)
print(f"\nH₀: La distribución de víctimas es igual entre todas las modalidades")
print(f"H₁: Al menos una modalidad tiene una distribución de víctimas diferente")
print(f"\nEstadístico H: {stat_kw:.2f}")
print(f"p-valor: {p_kw:.2e}")
print(f"Decisión: {'Se rechaza H₀' if p_kw < 0.05 else 'No se rechaza H₀'} (α = 0.05)")

if p_kw < 0.05:
    print(f"\n→ Existen diferencias estadísticamente significativas en la distribución")
    print(f"  de víctimas entre las distintas modalidades de acción bélica.")

    print(f"\n  Mediana de víctimas por modalidad:")
    medians = df_prep.groupby('Modalidad')['Total de Víctimas del Caso'].median().sort_values(ascending=False)
    for mod, med in medians.items():
        n = (df_prep['Modalidad'] == mod).sum()
        print(f"    {mod:<55} mediana = {med:.0f}  (n = {n:,})")

print(f"\n{'=' * 80}")
print("PRUEBA DE KRUSKAL-WALLIS: VÍCTIMAS POR GRUPO ARMADO 1")
print("=" * 80)

groups_ga1 = []
labels_ga1 = []
for ga, group in df_prep.groupby('Grupo Armado 1')['Total de Víctimas del Caso']:
    data = group.dropna().values
    if len(data) >= 5:
        groups_ga1.append(data)
        labels_ga1.append(ga)

stat_ga1, p_ga1 = kruskal(*groups_ga1)
print(f"\nH₀: La distribución de víctimas es igual entre todos los grupos armados")
print(f"H₁: Al menos un grupo armado tiene una distribución de víctimas diferente")
print(f"\nEstadístico H: {stat_ga1:.2f}")
print(f"p-valor: {p_ga1:.2e}")
print(f"Decisión: {'Se rechaza H₀' if p_ga1 < 0.05 else 'No se rechaza H₀'} (α = 0.05)")

print(f"\n{'=' * 80}")
print("PRUEBA DE MANN-WHITNEY U: VÍCTIMAS ANTES vs DESPUÉS DEL ACUERDO DE PAZ (2016)")
print("=" * 80)

df_valid_year = df_prep[df_prep['Año_int'] > 0].copy()
before_peace = df_valid_year[df_valid_year['Año_int'] < 2016]['Total de Víctimas del Caso'].dropna()
after_peace = df_valid_year[df_valid_year['Año_int'] >= 2016]['Total de Víctimas del Caso'].dropna()

stat_mw, p_mw = mannwhitneyu(before_peace, after_peace, alternative='two-sided')
print(f"\nH₀: La distribución de víctimas es igual antes y después de 2016")
print(f"H₁: Las distribuciones son diferentes")
print(f"\nAntes de 2016: n = {len(before_peace):,}, mediana = {before_peace.median():.0f}, media = {before_peace.mean():.2f}")
print(f"Desde 2016:    n = {len(after_peace):,}, mediana = {after_peace.median():.0f}, media = {after_peace.mean():.2f}")
print(f"\nEstadístico U: {stat_mw:.2f}")
print(f"p-valor: {p_mw:.2e}")
print(f"Decisión: {'Se rechaza H₀' if p_mw < 0.05 else 'No se rechaza H₀'} (α = 0.05)")
```

## 4. Regresión logística exploratoria

La entrega ajustó el modelo sobre **30.637 registros** y obtuvo **0,75 de exactitud en los mismos registros de ajuste**. Esa cifra describe ajuste interno. La estimación de desempeño para eventos futuros requeriría una separación temporal y codificación categórica revisada.

**Especificación y ajuste del modelo original**

```python
import statsmodels.api as sm
from sklearn.preprocessing import LabelEncoder

print("=" * 80)
print("1.2. REGRESIÓN LOGÍSTICA — Tiene_Víctimas")
print("=" * 80)

target_col = 'Tiene_Víctimas'
cat_features_reg = ['Modalidad', 'Grupo Armado 1', 'Iniciativa']
num_features_reg = ['Año_int']

df_reg = df_prep[cat_features_reg + num_features_reg + [target_col]].dropna().copy()
print(f"\nRegistros para el modelo (tras eliminar nulos): {len(df_reg):,}")

print(f"\nDistribución de la variable objetivo:")
vc = df_reg[target_col].value_counts()
for val, count in vc.items():
    print(f"  {target_col} = {val}: {count:,} ({count/len(df_reg)*100:.1f}%)")

df_reg = df_reg.reset_index(drop=True)
df_dummies = pd.get_dummies(df_reg[cat_features_reg], drop_first=True, dtype=int)
X = pd.concat([df_dummies, df_reg[num_features_reg]], axis=1)
X = sm.add_constant(X)
y = df_reg[target_col]

print(f"\nVariables predictoras en el modelo: {X.shape[1]} (incluyendo constante)")
print(f"  Categóricas codificadas (one-hot): {df_dummies.shape[1]}")
print(f"  Numéricas: {len(num_features_reg)}")

logit_model = sm.Logit(y, X)
logit_result = logit_model.fit(disp=0, maxiter=100)

print(f"\n{'=' * 80}")
print("RESUMEN DEL MODELO DE REGRESIÓN LOGÍSTICA")
print(f"{'=' * 80}")
print(logit_result.summary2())
```

**Umbral, matriz de confusión y reporte por clase**

```python
print("=" * 80)
print("EVALUACIÓN DEL MODELO")
print("=" * 80)

print(f"\nPseudo R² (McFadden): {logit_result.prsquared:.4f}")
print(f"Log-Likelihood: {logit_result.llf:.2f}")
print(f"AIC: {logit_result.aic:.2f}")
print(f"BIC: {logit_result.bic:.2f}")

y_pred_prob = logit_result.predict(X)
y_pred = (y_pred_prob >= 0.5).astype(int)
accuracy = (y_pred == y).mean()
print(f"Accuracy (umbral 0.5): {accuracy:.4f} ({accuracy*100:.1f}%)")

from sklearn.metrics import classification_report, confusion_matrix

cm = confusion_matrix(y, y_pred)
print(f"\nMatriz de confusión:")
print(f"  {'':>20} Predicho=0  Predicho=1")
print(f"  {'Real=0':<20} {cm[0,0]:>10,}  {cm[0,1]:>10,}")
print(f"  {'Real=1':<20} {cm[1,0]:>10,}  {cm[1,1]:>10,}")

print(f"\nReporte de clasificación:")
print(classification_report(y, y_pred, target_names=['Sin víctimas (0)', 'Con víctimas (1)']))

sig_vars = logit_result.pvalues[logit_result.pvalues < 0.05].index.tolist()
print(f"\nVariables significativas (p < 0.05): {len(sig_vars)}")
for var in sig_vars:
    coef = logit_result.params[var]
    p = logit_result.pvalues[var]
    odds_ratio = np.exp(coef)
    direction = "aumenta" if coef > 0 else "disminuye"
    print(f"  {var}: coef = {coef:.4f}, OR = {odds_ratio:.4f}, p = {p:.4e} "
          f"→ {direction} la probabilidad de víctimas")
```

## 5. Visualización territorial y comparación descriptiva

El mapa separa la intensidad registrada y la proporción de eventos con víctimas. Para comparar ventanas se expresa el conteo como eventos por año observado. El cambio alrededor de 2016 es descriptivo. La cobertura histórica, el registro desigual y el último año parcial limitan la comparación.

**Mapa de calor interactivo por actor y modalidad**

```python
import plotly.express as px
import plotly.graph_objects as go

df_heatmap = df_prep.groupby(['Grupo Armado 1', 'Modalidad']).agg(
    n_events=('Tiene_Víctimas', 'count'),
    tasa_victimas=('Tiene_Víctimas', 'mean'),
    total_victimas=('Total de Víctimas del Caso', 'sum')
).reset_index()

df_heatmap = df_heatmap[df_heatmap['Grupo Armado 1'] != 'OTROS']

fig_heat = px.density_heatmap(
    df_heatmap, x='Modalidad', y='Grupo Armado 1',
    z='tasa_victimas', histfunc='avg',
    color_continuous_scale='RdYlGn_r',
    title='Tasa de eventos con víctimas: Grupo Armado 1 × Modalidad',
    labels={'tasa_victimas': 'Tasa de víctimas', 'Modalidad': 'Modalidad', 'Grupo Armado 1': 'Grupo Armado 1'}
)
fig_heat.update_layout(width=900, height=600, xaxis_tickangle=-45)
fig_heat.show()

df_temporal = df_valid.groupby(['Año_int', 'Grupo Armado 1']).size().reset_index(name='n_events')
top_groups = df_prep['Grupo Armado 1'].value_counts().head(5).index.tolist()
df_temporal_top = df_temporal[df_temporal['Grupo Armado 1'].isin(top_groups)]

fig_line = px.line(
    df_temporal_top, x='Año_int', y='n_events', color='Grupo Armado 1',
    title='Evolución temporal de acciones bélicas por grupo armado principal',
    labels={'Año_int': 'Año', 'n_events': 'Número de eventos', 'Grupo Armado 1': 'Grupo Armado'}
)
fig_line.add_vline(x=2016, line_dash="dash", line_color="green", annotation_text="Acuerdo de Paz")
fig_line.update_layout(width=1000, height=500)
fig_line.show()
```

**Construcción del coroplético con Plotly**

```python
fig_choro1 = px.choropleth(
    dept_stats,
    geojson=colombia_geo,
    locations='Departamento_upper',
    featureidkey='id',
    color='n_eventos',
    color_continuous_scale='YlOrRd',
    scope='south america',
    hover_name='Departamento',
    hover_data={
        'Departamento_upper': False,
        'n_eventos': ':,',
        'tasa_victimas': ':.1%',
        'total_victimas': ':,',
        'media_victimas': ':.2f',
    },
    labels={
        'n_eventos': 'Total eventos',
        'tasa_victimas': 'Tasa con víctimas',
        'total_victimas': 'Total víctimas',
        'media_victimas': 'Media víctimas/evento',
    },
    title='Intensidad del conflicto armado por departamento — Número de acciones bélicas',
)
fig_choro1.update_geos(
    fitbounds='locations',
    visible=False,
    bgcolor='rgba(0,0,0,0)',
)
fig_choro1.update_layout(
    width=700, height=800,
    margin=dict(l=0, r=0, t=50, b=0),
    coloraxis_colorbar=dict(title='N° eventos', len=0.5),
)
fig_choro1.show()

fig_choro2 = px.choropleth(
    dept_stats,
    geojson=colombia_geo,
    locations='Departamento_upper',
    featureidkey='id',
    color='tasa_victimas',
    color_continuous_scale='RdYlGn_r',
    scope='south america',
    hover_name='Departamento',
    hover_data={
        'Departamento_upper': False,
        'n_eventos': ':,',
        'tasa_victimas': ':.1%',
        'total_victimas': ':,',
    },
    labels={
        'n_eventos': 'Total eventos',
        'tasa_victimas': 'Tasa con víctimas',
        'total_victimas': 'Total víctimas',
    },
    title='Tasa de eventos con víctimas por departamento',
)
fig_choro2.update_geos(fitbounds='locations', visible=False, bgcolor='rgba(0,0,0,0)')
fig_choro2.update_layout(
    width=700, height=800,
    margin=dict(l=0, r=0, t=50, b=0),
    coloraxis_colorbar=dict(title='Tasa víctimas', len=0.5, tickformat='.0%'),
)
fig_choro2.show()
```

**Normalización de recuentos por años observados**

```python
df_valid_yr = df_dept[df_dept['Año_int'] > 0].copy()

n_years_before = df_valid_yr[df_valid_yr['Año_int'] < 2016]['Año_int'].nunique()
n_years_after = df_valid_yr[df_valid_yr['Año_int'] >= 2016]['Año_int'].nunique()
print(f"Años distintos antes de 2016: {n_years_before}")
print(f"Años distintos desde 2016:    {n_years_after}")

def build_period_stats(df_src, periodo_label, n_years):
    agg = df_src.groupby('Departamento').agg(
        n_eventos=('Tiene_Víctimas', 'count'),
        tasa_victimas=('Tiene_Víctimas', 'mean'),
    ).reset_index()
    agg['eventos_por_año'] = (agg['n_eventos'] / n_years).round(1)
    agg['Departamento_upper'] = agg['Departamento'].str.upper().str.strip()
    agg['periodo'] = periodo_label
    return agg

df_before = build_period_stats(
    df_valid_yr[df_valid_yr['Año_int'] < 2016], 'Antes de 2016', n_years_before)
df_after = build_period_stats(
    df_valid_yr[df_valid_yr['Año_int'] >= 2016], 'Desde 2016', n_years_after)

df_periods = pd.concat([df_before, df_after], ignore_index=True)

fig_rate = px.choropleth(
    df_periods,
    geojson=colombia_geo,
    locations='Departamento_upper',
    featureidkey='id',
    color='eventos_por_año',
    animation_frame='periodo',
    color_continuous_scale='YlOrRd',
    scope='south america',
    hover_name='Departamento',
    hover_data={
        'Departamento_upper': False,
        'eventos_por_año': ':.1f',
        'n_eventos': ':,',
        'tasa_victimas': ':.1%',
    },
    labels={
        'eventos_por_año': 'Eventos/año',
        'n_eventos': 'Total eventos',
        'tasa_victimas': 'Tasa con víctimas',
    },
    title='Intensidad anualizada del conflicto por departamento: antes vs. después del Acuerdo de Paz',
    range_color=[0, df_periods['eventos_por_año'].quantile(0.95)],
)
fig_rate.update_geos(fitbounds='locations', visible=False, bgcolor='rgba(0,0,0,0)')
fig_rate.update_layout(
    width=750, height=850,
    margin=dict(l=0, r=0, t=60, b=0),
    coloraxis_colorbar=dict(title='Eventos/año', len=0.5),
)
fig_rate.show()

print("=" * 80)
print("RESUMEN COMPARATIVO — TASA ANUALIZADA (eventos/año)")
print("=" * 80)
for periodo, grupo in df_periods.groupby('periodo'):
    n_yrs = n_years_before if 'Antes' in periodo else n_years_after
    top5 = grupo.nlargest(5, 'eventos_por_año')[['Departamento', 'n_eventos', 'eventos_por_año', 'tasa_victimas']]
    print(f"\n  {periodo} ({n_yrs} años) — Top 5 departamentos:")
    for _, row in top5.iterrows():
        print(f"    {row['Departamento']:<25} {row['eventos_por_año']:>6.1f} eventos/año "
              f"(total: {row['n_eventos']:>5,}, tasa víctimas: {row['tasa_victimas']:.1%})")
```